# Transfer Learning — Aula 1: Conhecendo uma rede pré-treinada

## A ideia
Treinar uma CNN do zero (como no artigo) obriga a rede a reaprender até o básico — o que é uma borda, uma textura — a partir de apenas ~15 mil imagens. É um desperdício, e é parte do motivo de o modelo empacar em ~98%.

O **transfer learning** reaproveita uma rede já treinada em um dataset gigante (o **ImageNet**: 1,4 milhão de imagens). As camadas iniciais dessa rede já sabem enxergar bordas, formas e texturas — conhecimento que serve para qualquer imagem, raio-X incluso. Aproveitamos essa parte pronta e treinamos apenas uma "cabeça" nova para as nossas 4 classes, usando a nossa base de dados.

## Nesta aula
Antes de montar qualquer coisa, vamos apenas **carregar e inspecionar** a rede pré-treinada MobileNetV2, para enxergar na prática o que ela é: quantas camadas tem, quantos parâmetros, e qual é a saída original dela.

## 1. Carregando a MobileNetV2 pré-treinada

Usamos `keras.applications.MobileNetV2` para baixar a rede já treinada no ImageNet:
- `weights="imagenet"` → traz os pesos aprendidos nas 1,4 milhão de imagens (o "olho treinado").
- `include_top=True` → por enquanto, inclui a **cabeça original** da rede, aquela que classifica nas 1.000 categorias do ImageNet (gato, carro, cadeira...).

Na primeira vez, o Keras baixa os pesos da internet (~14 MB) e guarda no computador; nas próximas, usa a cópia local.

In [2]:
import tensorflow as tf
from tensorflow import keras

# Baixa a MobileNetV2 já treinada no ImageNet (na 1ª vez, baixa os pesos ~14 MB)
modelo_pretreinado = keras.applications.MobileNetV2(
    weights="imagenet",   # pesos aprendidos no ImageNet
    include_top=True      # inclui a cabeça original (1000 classes do ImageNet)
)

print("Número de camadas:", len(modelo_pretreinado.layers))
print("Total de parâmetros:", f"{modelo_pretreinado.count_params():,}")
print("Formato da saída:", modelo_pretreinado.output_shape)

E0000 00:00:1791591427.049159     734 cuda_platform.cc:52] failed call to cuInit: INTERNAL: CUDA error: Failed call to cuInit: UNKNOWN ERROR (303)


Número de camadas: 156
Total de parâmetros: 3,538,984
Formato da saída: (None, 1000)


## 2. Carregando a base pré-treinada (sem a cabeça) e congelando

- `include_top=False` → traz apenas o extrator de características, sem a cabeça de 1000 classes do ImageNet.
- `input_shape=(224, 224, 3)` → tamanho de entrada padrão da MobileNetV2 (ela foi treinada em imagens 224×224). Vamos redimensionar nossos raios-X para esse tamanho mais adiante.
- `base.trainable = False` → **congela** a base: seus pesos (aprendidos no ImageNet) não serão alterados durante o treino, preservando o conhecimento pré-treinado. Apenas a cabeça nova (criada depois) será treinada.

In [3]:
import numpy as np

# Carrega SÓ o extrator de características (sem a cabeça de 1000 classes)
base = keras.applications.MobileNetV2(
    weights="imagenet",
    include_top=False,
    input_shape=(224, 224, 3)
)

# Conta os parâmetros treináveis ANTES de congelar
treinaveis_antes = int(sum(np.prod(w.shape) for w in base.trainable_weights))

# CONGELA a base
base.trainable = False

# Conta de novo DEPOIS de congelar
treinaveis_depois = int(sum(np.prod(w.shape) for w in base.trainable_weights))

print("Formato da saída da base (sem cabeça):", base.output_shape)
print(f"Parâmetros treináveis ANTES de congelar:  {treinaveis_antes:,}")
print(f"Parâmetros treináveis DEPOIS de congelar: {treinaveis_depois:,}")

Formato da saída da base (sem cabeça): (None, 7, 7, 1280)
Parâmetros treináveis ANTES de congelar:  2,223,872
Parâmetros treináveis DEPOIS de congelar: 0


## 3. O preprocessamento próprio da MobileNetV2

Cada rede pré-treinada espera a entrada na mesma faixa de valores em que foi treinada. A MobileNetV2 **não** usa pixels de 0 a 1 (como o nosso `Rescaling(1./255)`), e sim de **−1 a +1**.

Por isso, em vez do `Rescaling`, usamos a função `preprocess_input` da própria MobileNetV2, que converte os pixels de 0–255 para a faixa −1 a +1. Abaixo, um exemplo mostra essa conversão.

In [4]:
from tensorflow.keras.applications.mobilenet_v2 import preprocess_input

# Três pixels de exemplo: o mínimo (0), o meio (127.5) e o máximo (255)
exemplo = np.array([[0.0, 127.5, 255.0]], dtype="float32")

print("Pixels originais (0 a 255):", exemplo)
print("Depois do preprocess_input:", preprocess_input(exemplo.copy()))

Pixels originais (0 a 255): [[  0.  127.5 255. ]]
Depois do preprocess_input: [[-1.  0.  1.]]


## 4. Montando o modelo: base congelada + cabeça nova

O modelo final tem três partes sobre a base:
- **Global Average Pooling (GAP):** em vez de "esticar" o mapa 7×7×1280 com Flatten (o que geraria 62.720 valores e muitos parâmetros), o GAP tira a média de cada canal, resumindo tudo em 1.280 valores. Menos parâmetros e menos overfitting.
- **Dropout(0.2):** regularização, desliga aleatoriamente 20% das conexões durante o treino.
- **Dense(4, softmax):** a cabeça de decisão nova, para as nossas 4 classes.

Como a base está congelada, apenas esta cabeça nova será treinada.

In [5]:
modelo = keras.Sequential([
    keras.layers.Input(shape=(224, 224, 3)),
    base,                                      # base pré-treinada e CONGELADA
    keras.layers.GlobalAveragePooling2D(),     # resume 7x7x1280 -> 1280
    keras.layers.Dropout(0.2),
    keras.layers.Dense(4, activation='softmax')  # cabeça nova: 4 classes
])

modelo.summary()

Model: "sequential"

┏━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━┓
┃ Layer (type)                    ┃ Output Shape           ┃       Param # ┃
┡━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━┩
│ mobilenetv2_1.00_224            │ (None, 7, 7, 1280)     │     2,257,984 │
│ (Functional)                    │                        │               │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ global_average_pooling2d_1      │ (None, 1280)           │             0 │
│ (GlobalAveragePooling2D)        │                        │               │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dropout (Dropout)               │ (None, 1280)           │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dense (Dense)                   │ (None, 4)              │         5,124 │
└─────────────────────────────────┴────────────────────────┴───────────────┘

 Total params: 2,263,108 (8.63 MB)

 Trainable params: 5,124 (20.02 KB)

 Non-trainable params: 2,257,984 (8.61 MB)